# Chunking Strategies

## Breaking long documents into searchable pieces

A researcher asks: "What was the economic impact of the Black Death on English agriculture?"

The answer is in our collection — specifically, in a 4,000-word research paper. But if we try to search the whole document, two things go wrong. First, the document is too long to be a useful search result — the researcher wants a specific passage, not the entire paper. Second, when we later send context to an LLM, we have a limited token budget, and we cannot afford to waste it on paragraphs about methodology and bibliography when the relevant content is in three paragraphs about labour economics.

We need to break documents into **chunks** — smaller pieces that can be independently searched and retrieved. But where we break them determines whether the search finds the answer.

Cut in the wrong place, and the answer spans two chunks. Neither chunk alone contains the complete thought. The search system returns half an answer, or worse, returns nothing at all.

## Loading a long document

In [ ]:
import json
import re

with open("../data/bl_research_papers.json", "r") as f:
    papers = json.load(f)

print(f"Loaded {len(papers)} research papers:")
for p in papers:
    print(f"  {p['paper_id']}: {p['title'][:60]}... ({len(p['text'])} chars, ~{len(p['text'].split())} words)")

In [ ]:
# Work with the first paper — the Black Death and English agriculture
paper = papers[0]
text = paper["text"]

print(f"Title: {paper['title']}")
print(f"Length: {len(text)} characters, {len(text.split())} words")
print(f"\nFirst 500 characters:\n{text[:500]}...")

This paper is far too long to be a single search result. We need to chunk it. Let us try three strategies and see how they compare.

## Strategy 1: Fixed-size chunking

The simplest approach: split every N characters. No regard for sentence boundaries, paragraph breaks, or document structure. Just cut.

In [ ]:
def chunk_fixed(text, chunk_size=500):
    """Split text into fixed-size chunks."""
    return [text[i:i+chunk_size] for i in range(0, len(text), chunk_size)]

fixed_chunks = chunk_fixed(text, chunk_size=500)
print(f"Fixed-size chunking: {len(fixed_chunks)} chunks of up to 500 characters")
print(f"\nChunk 3 (characters 1000-1500):")
print(repr(fixed_chunks[3]))

Look at that chunk boundary. It almost certainly cuts mid-sentence, possibly mid-word. The chunk starts partway through one thought and ends partway through another. Neither half makes complete sense on its own.

Fixed-size chunking is fast and simple, but it treats text as a sequence of bytes rather than a sequence of ideas. For search, this is a problem.

## Strategy 2: Sentence-based chunking

Split on sentence boundaries. Each chunk contains complete sentences, which means each chunk expresses complete thoughts.

In [ ]:
def chunk_by_sentences(text, max_chunk_size=500):
    """Split text into chunks at sentence boundaries, respecting max size."""
    # Split into sentences (handling common abbreviations)
    sentences = re.split(r'(?<=[.!?])\s+', text)
    
    chunks = []
    current_chunk = []
    current_length = 0
    
    for sentence in sentences:
        if current_length + len(sentence) > max_chunk_size and current_chunk:
            chunks.append(" ".join(current_chunk))
            current_chunk = []
            current_length = 0
        
        current_chunk.append(sentence)
        current_length += len(sentence) + 1  # +1 for space
    
    if current_chunk:
        chunks.append(" ".join(current_chunk))
    
    return chunks

sentence_chunks = chunk_by_sentences(text, max_chunk_size=500)
print(f"Sentence-based chunking: {len(sentence_chunks)} chunks")
print(f"\nChunk 3:")
print(sentence_chunks[3])

Better. Each chunk contains complete sentences. But we are still ignoring the document's own structure — its paragraphs and sections.

## Strategy 3: Paragraph-based chunking

Research papers and most long-form writing are organised into paragraphs. Each paragraph typically develops a single point or argument. Splitting on paragraph boundaries (`\n\n`) respects the author's own decisions about where one idea ends and another begins.

In [ ]:
def chunk_by_paragraphs(text, max_chunk_size=800, min_chunk_size=100):
    """Split text on paragraph boundaries, merging short paragraphs."""
    paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]
    
    chunks = []
    current_chunk = []
    current_length = 0
    
    for para in paragraphs:
        if current_length + len(para) > max_chunk_size and current_chunk:
            chunks.append("\n\n".join(current_chunk))
            current_chunk = []
            current_length = 0
        
        current_chunk.append(para)
        current_length += len(para)
    
    if current_chunk:
        chunk_text = "\n\n".join(current_chunk)
        # If the last chunk is too short, merge with previous
        if len(chunk_text) < min_chunk_size and chunks:
            chunks[-1] = chunks[-1] + "\n\n" + chunk_text
        else:
            chunks.append(chunk_text)
    
    return chunks

para_chunks = chunk_by_paragraphs(text, max_chunk_size=800)
print(f"Paragraph-based chunking: {len(para_chunks)} chunks")
print(f"\nChunk sizes: {[len(c) for c in para_chunks]}")
print(f"\nChunk 3:")
print(para_chunks[3])

Paragraph chunks are longer on average, but each one represents a coherent unit of thought. For search, this is often the best trade-off.

## Comparing the three strategies

In [ ]:
import pandas as pd

strategies = {
    "Fixed (500 chars)": fixed_chunks,
    "Sentence-based": sentence_chunks,
    "Paragraph-based": para_chunks,
}

comparison = []
for name, chunks in strategies.items():
    sizes = [len(c) for c in chunks]
    comparison.append({
        "Strategy": name,
        "Num chunks": len(chunks),
        "Avg size (chars)": int(sum(sizes) / len(sizes)),
        "Min size": min(sizes),
        "Max size": max(sizes),
    })

pd.DataFrame(comparison)

## The overlap problem

Here is the scenario that motivates everything that follows.

A researcher asks about the economic consequences of the Black Death. The answer spans the boundary between two chunks. The first chunk ends with a sentence about rising wages; the second begins with a sentence about falling rents. Together, they answer the question. Separately, neither does.

Let us find a concrete example.

In [ ]:
# Find a chunk boundary where the answer to a question might be split
# Look at consecutive sentence-based chunks
query = "labour shortage wages"

print("Searching for relevant content across chunk boundaries...\n")

for i in range(len(sentence_chunks) - 1):
    chunk_a = sentence_chunks[i].lower()
    chunk_b = sentence_chunks[i + 1].lower()
    
    # Check if query terms are split across boundary
    query_terms = query.lower().split()
    terms_in_a = sum(1 for t in query_terms if t in chunk_a)
    terms_in_b = sum(1 for t in query_terms if t in chunk_b)
    
    if terms_in_a >= 1 and terms_in_b >= 1:
        print(f"Chunk boundary {i}/{i+1} splits relevant content:")
        print(f"\n--- End of chunk {i} ---")
        print(sentence_chunks[i][-200:])
        print(f"\n--- Start of chunk {i+1} ---")
        print(sentence_chunks[i+1][:200])
        print()
        break

The solution is **overlap**. Instead of cutting cleanly between chunks, we let adjacent chunks share some content. The end of chunk N overlaps with the beginning of chunk N+1. This way, information near a boundary appears in both chunks.

In [ ]:
def chunk_with_overlap(text, chunk_size=500, overlap=100):
    """Split text into chunks with overlap, respecting sentence boundaries."""
    sentences = re.split(r'(?<=[.!?])\s+', text)
    
    chunks = []
    current_chunk = []
    current_length = 0
    
    for sentence in sentences:
        if current_length + len(sentence) > chunk_size and current_chunk:
            chunks.append(" ".join(current_chunk))
            
            # Keep the last few sentences as overlap
            overlap_chunk = []
            overlap_len = 0
            for s in reversed(current_chunk):
                if overlap_len + len(s) <= overlap:
                    overlap_chunk.insert(0, s)
                    overlap_len += len(s)
                else:
                    break
            
            current_chunk = overlap_chunk
            current_length = overlap_len
        
        current_chunk.append(sentence)
        current_length += len(sentence) + 1
    
    if current_chunk:
        chunks.append(" ".join(current_chunk))
    
    return chunks

overlap_chunks = chunk_with_overlap(text, chunk_size=500, overlap=100)
print(f"Chunks with overlap: {len(overlap_chunks)}")
print(f"Chunks without overlap: {len(sentence_chunks)}")
print(f"\nThe overlap creates {len(overlap_chunks) - len(sentence_chunks)} additional chunks.")

In [ ]:
# Verify that overlapping chunks share content
if len(overlap_chunks) >= 3:
    print("--- End of chunk 1 ---")
    print(overlap_chunks[1][-150:])
    print("\n--- Start of chunk 2 ---")
    print(overlap_chunks[2][:150])
    
    # Check for shared text
    shared = set(overlap_chunks[1].split()) & set(overlap_chunks[2].split())
    print(f"\nShared words between adjacent chunks: {len(shared)}")

## How much overlap?

More overlap means better coverage of boundary-spanning information, but also more redundancy. Each overlapping token is stored and searched twice.

Typical rules of thumb:
- **10-20% overlap** is a reasonable starting point
- For question-answering, where answers tend to be 1-3 sentences, 50-100 tokens of overlap usually suffices
- For summarisation, where you need broader context, you might want more

Let us measure the trade-off.

In [ ]:
# Compare different overlap amounts
overlaps = [0, 50, 100, 150, 200, 250]
results = []

for overlap in overlaps:
    chunks = chunk_with_overlap(text, chunk_size=500, overlap=overlap)
    total_chars = sum(len(c) for c in chunks)
    redundancy = (total_chars - len(text)) / len(text)
    results.append({
        "Overlap (chars)": overlap,
        "Num chunks": len(chunks),
        "Total chars stored": total_chars,
        "Redundancy": f"{redundancy:.1%}",
    })

pd.DataFrame(results)

There is a real cost to overlap. Doubling the overlap roughly doubles the redundancy. For a collection of millions of documents, that redundancy translates to significant storage and computation. Choose your overlap based on your use case and your budget.

## Metadata preservation

A chunk on its own is useless if you cannot trace it back to its source. Every chunk should carry metadata: which document it came from, where in the document it sits, and any other information a search system might need.

In [ ]:
def chunk_document(paper, chunk_size=500, overlap=100):
    """Chunk a paper and attach metadata to each chunk."""
    text = paper["text"]
    raw_chunks = chunk_with_overlap(text, chunk_size=chunk_size, overlap=overlap)
    
    enriched_chunks = []
    for idx, chunk_text in enumerate(raw_chunks):
        # Find approximate position in original text
        start_pos = text.find(chunk_text[:50])
        
        enriched_chunks.append({
            "chunk_id": f"{paper['paper_id']}-C{idx+1:03d}",
            "paper_id": paper["paper_id"],
            "paper_title": paper["title"],
            "author": paper["author"],
            "year": paper["year"],
            "chunk_index": idx,
            "total_chunks": len(raw_chunks),
            "chunk_text": chunk_text,
            "char_start": start_pos if start_pos >= 0 else None,
            "chunk_length": len(chunk_text),
        })
    
    return enriched_chunks

# Chunk all papers
all_chunks = []
for paper in papers:
    chunks = chunk_document(paper, chunk_size=500, overlap=100)
    all_chunks.extend(chunks)

chunks_df = pd.DataFrame(all_chunks)
print(f"Total chunks across all papers: {len(chunks_df)}")
print(f"\nChunks per paper:")
print(chunks_df.groupby("paper_title")["chunk_id"].count().to_string())

In [ ]:
# Inspect a chunk with its metadata
sample_chunk = chunks_df.iloc[5]
print(f"Chunk ID:    {sample_chunk['chunk_id']}")
print(f"Paper:       {sample_chunk['paper_title'][:60]}...")
print(f"Author:      {sample_chunk['author']}")
print(f"Position:    chunk {sample_chunk['chunk_index']+1} of {sample_chunk['total_chunks']}")
print(f"Length:      {sample_chunk['chunk_length']} chars")
print(f"\nText:\n{sample_chunk['chunk_text'][:300]}...")

## Chunk quality assessment

Not all chunks are equally useful. Some are too short (a single sentence with no context), some are too long (half a paper crammed into one chunk), and some hit the sweet spot — enough context to be meaningful, focused enough to be relevant.

Let us define some quality criteria.

In [ ]:
def assess_chunk_quality(chunk_text):
    """Assess the quality of a chunk. Returns a dict of quality metrics."""
    words = chunk_text.split()
    sentences = re.split(r'(?<=[.!?])\s+', chunk_text)
    
    return {
        "char_count": len(chunk_text),
        "word_count": len(words),
        "sentence_count": len(sentences),
        "avg_sentence_length": len(words) / max(len(sentences), 1),
        "too_short": len(words) < 20,
        "too_long": len(words) > 300,
        "starts_mid_sentence": not chunk_text[0].isupper() if chunk_text else True,
        "ends_mid_sentence": not chunk_text.rstrip().endswith(".") if chunk_text else True,
    }

# Assess all chunks
quality = pd.DataFrame([assess_chunk_quality(c) for c in chunks_df["chunk_text"]])

print("Chunk quality summary:")
print(f"  Too short (<20 words):       {quality['too_short'].sum()}")
print(f"  Too long (>300 words):        {quality['too_long'].sum()}")
print(f"  Starts mid-sentence:          {quality['starts_mid_sentence'].sum()}")
print(f"  Ends mid-sentence:            {quality['ends_mid_sentence'].sum()}")
print(f"\n  Average words per chunk:      {quality['word_count'].mean():.0f}")
print(f"  Average sentences per chunk:  {quality['sentence_count'].mean():.1f}")

## A configurable chunking function

Let us wrap everything into a single function that supports multiple strategies.

In [ ]:
def chunk_text(text, strategy="sentence", chunk_size=500, overlap=100, 
               min_chunk_size=50):
    """Chunk text using the specified strategy.
    
    Parameters
    ----------
    text : str
        The text to chunk.
    strategy : str
        One of 'fixed', 'sentence', or 'paragraph'.
    chunk_size : int
        Target chunk size in characters.
    overlap : int
        Number of characters to overlap between adjacent chunks.
    min_chunk_size : int
        Minimum chunk size; shorter chunks are merged with the previous.
    """
    if strategy == "fixed":
        # No overlap for fixed-size — it would break mid-word
        chunks = [text[i:i+chunk_size] for i in range(0, len(text), chunk_size)]
    
    elif strategy == "sentence":
        chunks = chunk_with_overlap(text, chunk_size=chunk_size, overlap=overlap)
    
    elif strategy == "paragraph":
        paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]
        chunks = []
        current = []
        current_len = 0
        
        for para in paragraphs:
            if current_len + len(para) > chunk_size and current:
                chunks.append("\n\n".join(current))
                # Overlap: keep last paragraph if it fits
                if len(current[-1]) <= overlap:
                    current = [current[-1]]
                    current_len = len(current[-1])
                else:
                    current = []
                    current_len = 0
            current.append(para)
            current_len += len(para)
        
        if current:
            chunk_text_str = "\n\n".join(current)
            if len(chunk_text_str) < min_chunk_size and chunks:
                chunks[-1] += "\n\n" + chunk_text_str
            else:
                chunks.append(chunk_text_str)
    
    else:
        raise ValueError(f"Unknown strategy: {strategy}. Use 'fixed', 'sentence', or 'paragraph'.")
    
    # Filter out empty chunks
    chunks = [c for c in chunks if c.strip()]
    
    return chunks

In [ ]:
# Compare all three strategies on every paper
comparison_rows = []

for paper in papers:
    for strategy in ["fixed", "sentence", "paragraph"]:
        chunks = chunk_text(paper["text"], strategy=strategy, chunk_size=500, overlap=100)
        sizes = [len(c) for c in chunks]
        comparison_rows.append({
            "Paper": paper["paper_id"],
            "Strategy": strategy,
            "Chunks": len(chunks),
            "Avg size": int(sum(sizes) / len(sizes)),
            "Min": min(sizes),
            "Max": max(sizes),
        })

pd.DataFrame(comparison_rows)

---

## Exercises

### Exercise 1: Section-aware chunking

The research papers have section headings (e.g., "Introduction", "The Pre-Plague Agricultural Economy"). Write a chunking function that respects section boundaries — never splits across a section heading. You can detect headings as short lines (fewer than 80 characters) that appear after a blank line.

Compare the results with the paragraph-based strategy.

In [ ]:
# Your code here


### Exercise 2: Find the worst chunk boundary

Using the sentence-based chunks (no overlap) of the Black Death paper, find the chunk boundary where the most relevant information about "labour shortage" is split across two adjacent chunks. Print the last 100 characters of one chunk and the first 100 characters of the next.

Then re-chunk with 150 characters of overlap and show that the information now appears in a single chunk.

In [ ]:
# Your code here


### Exercise 3: Chunk size optimisation

For a single research paper, generate chunks at sizes from 200 to 1000 characters (step 100), all with the sentence-based strategy and 20% overlap (i.e., overlap = 0.2 * chunk_size).

For each size, compute:
- Number of chunks
- Average chunk word count
- Percentage of chunks that start or end mid-sentence
- Total storage (sum of all chunk sizes)

Plot or tabulate the results. What chunk size would you recommend?

In [ ]:
# Your code here


### Your turn: Chunk the full collection

Using the chunking strategy and parameters of your choice, chunk all five research papers. Store the results in a DataFrame with columns: `chunk_id`, `paper_id`, `chunk_index`, `chunk_text`, `word_count`.

How many chunks do you get in total? What is the distribution of chunk sizes?

In [ ]:
# Your code here


---

## Summary

In this notebook we explored three chunking strategies:

1. **Fixed-size** — fast and simple, but cuts mid-sentence and mid-thought
2. **Sentence-based** — respects sentence boundaries, much better for search
3. **Paragraph-based** — respects document structure, best for coherent chunks

We also covered:
- **Overlap** — sharing content between adjacent chunks prevents information loss at boundaries
- **Metadata preservation** — every chunk carries its source document ID, position, and context
- **Quality assessment** — not all chunks are equally useful; measure before you commit

The big idea: where you cut matters as much as how small you cut. A chunk boundary through the middle of an answer is worse than no chunking at all. Overlap is the safety net.

Next up: we have clean, deduplicated, chunked text. Now we need to make it searchable. Keyword search will not be enough — we need to understand meaning, not just match words.